# Mental Health R1 Human Validation Audit and Supplementary Table S2

This notebook is configured for the anonymized review repository. Download the OSF review data package and place it under `0. Dataset/` before running. It does not call external APIs and does not require author-local paths.

It produces:

- human-validation QC checks for both annotators;
- primary-label agreement, triad, and aspect-agreement tables for the R1 manuscript;
- row-level comparison files for auditability;
- Supplementary Table S2 from the original-vs-AI confusion-matrix logic.


In [1]:
# Review repository setup
# Place the OSF review data package under ../0. Dataset/ relative to this notebook,
# or set the DATA_ROOT environment variable before running.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import re
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp')

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


In [3]:
# -----------------------------
# Path configuration
# -----------------------------

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.startswith('6. Revision'):
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = Path(os.environ.get('DATA_ROOT', REPO_ROOT / '0. Dataset'))
OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', REPO_ROOT / 'outputs'))

# Expected after downloading the OSF review data package:
# 0. Dataset/human_validation_300/generated_subset/human_validation_sample_key_300.csv
# 0. Dataset/human_validation_300/outputs/human_validation_annotator1_300.xlsm
# 0. Dataset/human_validation_300/outputs/human_validation_annotator2_300.xlsm
# 0. Dataset/labeling/mental_health_unified_labels_final.csv
HUMAN_DIR = DATA_ROOT / 'human_validation_300'
LABELING_DIR = DATA_ROOT / 'labeling'

SAMPLE_KEY_PATH = HUMAN_DIR / 'generated_subset' / 'human_validation_sample_key_300.csv'
ANNOTATOR1_PATH = HUMAN_DIR / 'outputs' / 'human_validation_annotator1_300.xlsm'
ANNOTATOR2_PATH = HUMAN_DIR / 'outputs' / 'human_validation_annotator2_300.xlsm'
LABEL_SOURCE_PATH = LABELING_DIR / 'mental_health_unified_labels_final.csv'

OUT_DIR = OUTPUT_ROOT / 'human_validation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [SAMPLE_KEY_PATH, ANNOTATOR1_PATH, ANNOTATOR2_PATH, LABEL_SOURCE_PATH]:
    if not p.exists():
        print(f'Missing expected input: {p}')

SAMPLE_KEY_PATH, ANNOTATOR1_PATH, ANNOTATOR2_PATH, LABEL_SOURCE_PATH, OUT_DIR


BASE_ROOT: ./ | exists=True
HUMAN_DIR: ./02_R1/01_human_validation_300 | exists=True
LABELING_DIR: ./01_R0/00_Labelling | exists=True
SAMPLE_KEY_PATH: ./02_R1/01_human_validation_300/generated_subset/human_validation_sample_key_300.csv | exists=True
ANNOTATOR1_PATH: ./02_R1/01_human_validation_300/outputs/human_validation_annotator1_300.xlsm | exists=True
ANNOTATOR2_PATH: ./02_R1/01_human_validation_300/outputs/human_validation_annotator2_300.xlsm | exists=True
LABEL_SOURCE_PATH: ./01_R0/00_Labelling/data/mental_health_unified_labels_final.csv | exists=True
OUT_DIR: ./02_R1/01_human_validation_300/outputs_notebook | exists=True


In [4]:
# -----------------------------
# Constants and normalization helpers
# -----------------------------

LABELS = [
    'NORMAL',
    'DEPRESSION',
    'ANXIETY',
    'SUICIDAL',
    'STRESS',
    'BIPOLAR',
    'PERSONALITY_DISORDER',
]

LABEL_DISPLAY = {
    'NORMAL': 'Normal',
    'DEPRESSION': 'Depression',
    'ANXIETY': 'Anxiety',
    'SUICIDAL': 'Suicidal',
    'STRESS': 'Stress',
    'BIPOLAR': 'Bipolar',
    'PERSONALITY_DISORDER': 'Personality Disorder',
}

S2_DISPLAY = {
    'NORMAL': 'Normal',
    'DEPRESSION': 'Depression',
    'ANXIETY': 'Anxiety',
    'SUICIDAL': 'Suicidal',
    'STRESS': 'Stress',
    'BIPOLAR': 'Bipolar',
    'PERSONALITY_DISORDER': 'Pers. Dis.',
}

ASPECTS = [
    'depression',
    'anxiety',
    'suicidal',
    'stress',
    'bipolar',
    'personality_disorder',
]

ASPECT_DISPLAY = {
    'depression': 'Depression',
    'anxiety': 'Anxiety',
    'suicidal': 'Suicidal',
    'stress': 'Stress',
    'bipolar': 'Bipolar',
    'personality_disorder': 'Personality Disorder',
}

PRIMARY_TO_ASPECT = {
    'DEPRESSION': 'depression',
    'ANXIETY': 'anxiety',
    'SUICIDAL': 'suicidal',
    'STRESS': 'stress',
    'BIPOLAR': 'bipolar',
    'PERSONALITY_DISORDER': 'personality_disorder',
}

STRENGTHS = ['none', 'weak', 'clear']
POSITIVE_STRENGTHS = {'weak', 'clear'}

def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip().lower()


def canonical_label(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip().upper().replace('-', '_').replace(' ', '_')
    text = re.sub(r'_+', '_', text)
    aliases = {
        'PERSONALITY': 'PERSONALITY_DISORDER',
        'PERSONALITY_DIS': 'PERSONALITY_DISORDER',
        'PERSONALITY_DIS.': 'PERSONALITY_DISORDER',
        'PERSONALITY_DISORDER': 'PERSONALITY_DISORDER',
        'PERSONALITYDISORDER': 'PERSONALITY_DISORDER',
        'PERS_DIS': 'PERSONALITY_DISORDER',
        'PERS_DIS.': 'PERSONALITY_DISORDER',
        'PERS._DIS.': 'PERSONALITY_DISORDER',
        'NONE': 'NORMAL',
    }
    return aliases.get(text, text)


def display_label(value: object, mapping: dict[str, str] = LABEL_DISPLAY) -> str:
    lab = canonical_label(value)
    return mapping.get(lab, lab)


def canonical_strength(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    aliases = {
        'no': 'none',
        '0': 'none',
        '1': 'weak',
        '2': 'clear',
        'nan': '',
    }
    return aliases.get(text, text)


def binary_presence(value: object) -> str:
    return 'present' if canonical_strength(value) in POSITIVE_STRENGTHS else 'absent'


def pct(n, d):
    return np.nan if d == 0 else 100.0 * float(n) / float(d)


In [5]:
# -----------------------------
# Load key and annotator workbooks
# -----------------------------

def read_annotation_file(path: Path, annotator: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {'.xlsx', '.xlsm', '.xls'}:
        df = pd.read_excel(path, sheet_name='Annotation', engine='openpyxl')
    else:
        df = pd.read_csv(path)

    df = df.copy()
    df = df[df['sample_id'].notna()].copy()
    df['sample_id'] = df['sample_id'].astype(str).str.strip()
    df[f'{annotator}_statement'] = df['statement']
    df[f'{annotator}_statement_norm'] = df['statement'].map(normalize_text)
    df[f'{annotator}_primary'] = df['human_primary_label'].map(canonical_label)
    df[f'{annotator}_uncertain'] = df.get('human_uncertain', '').map(lambda x: str(x).strip().lower() if not pd.isna(x) else '')
    df[f'{annotator}_notes'] = df.get('human_notes', '')
    for aspect in ASPECTS:
        human_col = f'human_{aspect}'
        df[f'{annotator}_{aspect}'] = df[human_col].map(canonical_strength) if human_col in df.columns else ''
    keep = ['sample_id', f'{annotator}_statement', f'{annotator}_statement_norm', f'{annotator}_primary', f'{annotator}_uncertain', f'{annotator}_notes']
    keep += [f'{annotator}_{aspect}' for aspect in ASPECTS]
    return df[keep]


def read_sample_key(path: Path) -> pd.DataFrame:
    key = pd.read_csv(path, low_memory=False)
    key = key.copy()
    key['sample_id'] = key['sample_id'].astype(str).str.strip()
    key['key_statement_norm'] = key['statement'].map(normalize_text)
    key['released_label'] = key['status'].map(canonical_label)
    key['ai_label'] = key['u_label'].map(canonical_label)
    for aspect in ASPECTS:
        col = f'u_{aspect}_strength'
        key[f'ai_{aspect}'] = key[col].map(canonical_strength) if col in key.columns else 'none'
    return key


def build_audit_frame(sample_key_path: Path, annotator1_path: Path, annotator2_path: Path) -> pd.DataFrame:
    key = read_sample_key(sample_key_path)
    a1 = read_annotation_file(annotator1_path, 'a1')
    a2 = read_annotation_file(annotator2_path, 'a2')
    frame = key.merge(a1, on='sample_id', how='left').merge(a2, on='sample_id', how='left')
    frame['a1_statement_match'] = frame['key_statement_norm'] == frame['a1_statement_norm']
    frame['a2_statement_match'] = frame['key_statement_norm'] == frame['a2_statement_norm']
    return frame

frame = build_audit_frame(SAMPLE_KEY_PATH, ANNOTATOR1_PATH, ANNOTATOR2_PATH)
print('Audit rows:', len(frame))
print('Columns:', len(frame.columns))
frame[['sample_id', 'released_label', 'ai_label', 'a1_primary', 'a2_primary', 'a1_statement_match', 'a2_statement_match']].head()


Audit rows: 300
Columns: 76


,sample_id,released_label,ai_label,a1_primary,a2_primary,a1_statement_match,a2_statement_match
0,MHV_0001,ANXIETY,SUICIDAL,SUICIDAL,SUICIDAL,True,True
1,MHV_0002,PERSONALITY_DISORDER,ANXIETY,SUICIDAL,SUICIDAL,True,True
2,MHV_0003,SUICIDAL,ANXIETY,SUICIDAL,SUICIDAL,True,True
3,MHV_0004,STRESS,STRESS,STRESS,STRESS,True,True
4,MHV_0005,BIPOLAR,STRESS,STRESS,STRESS,True,True


In [6]:
# -----------------------------
# Quality control checks
# -----------------------------

def annotator_qc(frame: pd.DataFrame, annotator: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    issues = []
    primary_col = f'{annotator}_primary'
    statement_match_col = f'{annotator}_statement_match'
    aspect_cols = [f'{annotator}_{aspect}' for aspect in ASPECTS]

    completed_primary = frame[primary_col].isin(LABELS)
    rows.append({'annotator': annotator, 'check': 'rows_with_valid_primary_label', 'n': int(completed_primary.sum()), 'denominator': int(len(frame))})

    invalid_primary = frame[~frame[primary_col].isin(LABELS)]
    rows.append({'annotator': annotator, 'check': 'invalid_primary_label_rows', 'n': int(len(invalid_primary)), 'denominator': int(len(frame))})
    for _, r in invalid_primary.iterrows():
        issues.append({'annotator': annotator, 'sample_id': r['sample_id'], 'issue': 'invalid_primary_label', 'value': r.get(primary_col, '')})

    invalid_strength_count = 0
    for aspect, col in zip(ASPECTS, aspect_cols):
        invalid = frame[~frame[col].isin(STRENGTHS)]
        invalid_strength_count += len(invalid)
        for _, r in invalid.iterrows():
            issues.append({'annotator': annotator, 'sample_id': r['sample_id'], 'issue': f'invalid_{aspect}_strength', 'value': r.get(col, '')})
    rows.append({'annotator': annotator, 'check': 'invalid_aspect_strength_cells', 'n': int(invalid_strength_count), 'denominator': int(len(frame) * len(ASPECTS))})

    positive_any = frame[aspect_cols].isin(POSITIVE_STRENGTHS).any(axis=1)
    normal_positive = frame[(frame[primary_col] == 'NORMAL') & positive_any]
    rows.append({'annotator': annotator, 'check': 'normal_rows_with_positive_aspect', 'n': int(len(normal_positive)), 'denominator': int(len(frame))})
    for _, r in normal_positive.iterrows():
        issues.append({'annotator': annotator, 'sample_id': r['sample_id'], 'issue': 'normal_with_positive_aspect', 'value': ''})

    missing_corresponding = []
    for _, r in frame.iterrows():
        primary = r.get(primary_col, '')
        aspect = PRIMARY_TO_ASPECT.get(primary)
        if aspect and r.get(f'{annotator}_{aspect}', '') not in POSITIVE_STRENGTHS:
            missing_corresponding.append(r)
            issues.append({'annotator': annotator, 'sample_id': r['sample_id'], 'issue': 'non_normal_primary_missing_corresponding_aspect', 'value': primary})
    rows.append({'annotator': annotator, 'check': 'non_normal_primary_missing_corresponding_aspect', 'n': int(len(missing_corresponding)), 'denominator': int(len(frame))})

    statement_mismatch = frame[~frame[statement_match_col].fillna(False)]
    rows.append({'annotator': annotator, 'check': 'statement_mismatch_rows', 'n': int(len(statement_mismatch)), 'denominator': int(len(frame))})
    for _, r in statement_mismatch.iterrows():
        issues.append({'annotator': annotator, 'sample_id': r['sample_id'], 'issue': 'statement_mismatch_vs_key', 'value': ''})

    return pd.DataFrame(rows), pd.DataFrame(issues)

qc1, issues1 = annotator_qc(frame, 'a1')
qc2, issues2 = annotator_qc(frame, 'a2')
qc_summary = pd.concat([qc1, qc2], ignore_index=True)
qc_issues = pd.concat([issues1, issues2], ignore_index=True)

qc_summary.to_csv(OUT_DIR / 'human_validation_qc_summary.csv', index=False)
qc_issues.to_csv(OUT_DIR / 'human_validation_qc_issues.csv', index=False)

qc_summary


,annotator,check,n,denominator
0,a1,rows_with_valid_primary_label,300,300
1,a1,invalid_primary_label_rows,0,300
2,a1,invalid_aspect_strength_cells,0,1800
3,a1,normal_rows_with_positive_aspect,0,300
4,a1,non_normal_primary_missing_corresponding_aspect,0,300
5,a1,statement_mismatch_rows,0,300
6,a2,rows_with_valid_primary_label,300,300
7,a2,invalid_primary_label_rows,0,300
8,a2,invalid_aspect_strength_cells,0,1800
9,a2,normal_rows_with_positive_aspect,0,300


In [7]:
qc_issues


,annotator,sample_id,issue,value
0,a2,MHV_0252,statement_mismatch_vs_key,


In [8]:
# -----------------------------
# Agreement tables
# -----------------------------

def pairwise_primary_agreement(frame: pd.DataFrame, left_col: str, right_col: str, label: str) -> dict[str, object]:
    use = frame[[left_col, right_col]].dropna().copy()
    use = use[(use[left_col] != '') & (use[right_col] != '')]
    n = int(len(use))
    agree = int((use[left_col] == use[right_col]).sum()) if n else 0
    kappa = float(cohen_kappa_score(use[left_col], use[right_col], labels=LABELS)) if n else np.nan
    return {
        'comparison': label,
        'exact_agreement': f'{agree}/{n}',
        'agreement_n': agree,
        'n': n,
        'agreement_pct': round(pct(agree, n), 1),
        'cohen_kappa': round(kappa, 3) if not np.isnan(kappa) else np.nan,
    }

primary_agreement = pd.DataFrame([
    pairwise_primary_agreement(frame, 'a1_primary', 'a2_primary', 'Annotator 1 vs. Annotator 2'),
    pairwise_primary_agreement(frame, 'a1_primary', 'ai_label', 'Annotator 1 vs. AI label'),
    pairwise_primary_agreement(frame, 'a2_primary', 'ai_label', 'Annotator 2 vs. AI label'),
    pairwise_primary_agreement(frame, 'a1_primary', 'released_label', 'Annotator 1 vs. released label'),
    pairwise_primary_agreement(frame, 'a2_primary', 'released_label', 'Annotator 2 vs. released label'),
    pairwise_primary_agreement(frame, 'ai_label', 'released_label', 'AI label vs. released label'),
])
primary_agreement.to_csv(OUT_DIR / 'human_validation_primary_agreement.csv', index=False)
primary_agreement


,comparison,exact_agreement,agreement_n,n,agreement_pct,cohen_kappa
0,Annotator 1 vs. Annotator 2,169/300,169,300,56.3,0.489
1,Annotator 1 vs. AI label,155/300,155,300,51.7,0.436
2,Annotator 2 vs. AI label,164/300,164,300,54.7,0.462
3,Annotator 1 vs. released label,158/300,158,300,52.7,0.449
4,Annotator 2 vs. released label,145/300,145,300,48.3,0.401
5,AI label vs. released label,138/300,138,300,46.0,0.373


In [9]:
def primary_confusion(frame: pd.DataFrame, row_col: str, col_col: str) -> pd.DataFrame:
    table = pd.crosstab(frame[row_col], frame[col_col])
    table = table.reindex(index=LABELS, columns=LABELS, fill_value=0)
    table.index = [LABEL_DISPLAY[x] for x in table.index]
    table.columns = [LABEL_DISPLAY[x] for x in table.columns]
    return table

confusion_specs = {
    'a1_vs_a2': ('a1_primary', 'a2_primary'),
    'a1_vs_ai': ('a1_primary', 'ai_label'),
    'a2_vs_ai': ('a2_primary', 'ai_label'),
    'a1_vs_released': ('a1_primary', 'released_label'),
    'a2_vs_released': ('a2_primary', 'released_label'),
    'ai_vs_released': ('ai_label', 'released_label'),
}

for name, (row_col, col_col) in confusion_specs.items():
    primary_confusion(frame, row_col, col_col).to_csv(OUT_DIR / f'human_validation_confusion_{name}.csv')

primary_confusion(frame, 'a1_primary', 'a2_primary')


,Normal,Depression,Anxiety,Suicidal,Stress,Bipolar,Personality Disorder
Normal,39,9,12,0,4,0,2
Depression,3,19,13,2,3,0,1
Anxiety,2,1,22,1,7,0,0
Suicidal,2,6,7,27,3,7,1
Stress,9,3,8,2,19,0,0
Bipolar,0,2,2,0,1,25,1
Personality Disorder,2,2,11,0,2,0,18


In [10]:
# -----------------------------
# Three-way primary-label patterns
# -----------------------------

def triad_counts(frame: pd.DataFrame, human_col: str, human_name: str) -> pd.DataFrame:
    rows = []
    n = len(frame)
    h = frame[human_col]
    ai = frame['ai_label']
    rel = frame['released_label']
    patterns = [
        ('Human = AI = released', (h == ai) & (ai == rel)),
        ('Human = released != AI', (h == rel) & (h != ai)),
        ('Human = AI != released', (h == ai) & (h != rel)),
        ('AI = released != Human', (ai == rel) & (h != ai)),
        ('All three differ', (h != ai) & (h != rel) & (ai != rel)),
    ]
    for label, mask in patterns:
        count = int(mask.sum())
        rows.append({'human_reference': human_name, 'pattern': label, 'n': count, 'pct': round(pct(count, n), 1), 'denominator': n})
    return pd.DataFrame(rows)

triad = pd.concat([
    triad_counts(frame, 'a1_primary', 'Annotator 1'),
    triad_counts(frame, 'a2_primary', 'Annotator 2'),
], ignore_index=True)
triad.to_csv(OUT_DIR / 'human_validation_triad_counts.csv', index=False)
triad


,human_reference,pattern,n,pct,denominator
0,Annotator 1,Human = AI = released,100,33.3,300
1,Annotator 1,Human = released != AI,58,19.3,300
2,Annotator 1,Human = AI != released,55,18.3,300
3,Annotator 1,AI = released != Human,38,12.7,300
4,Annotator 1,All three differ,49,16.3,300
5,Annotator 2,Human = AI = released,97,32.3,300
6,Annotator 2,Human = released != AI,48,16.0,300
7,Annotator 2,Human = AI != released,67,22.3,300
8,Annotator 2,AI = released != Human,41,13.7,300
9,Annotator 2,All three differ,47,15.7,300


In [11]:
# -----------------------------
# Aspect agreement
# -----------------------------

def aspect_pair_agreement(frame: pd.DataFrame, left_prefix: str, right_prefix: str, comparison: str) -> tuple[dict[str, object], pd.DataFrame]:
    by_aspect = []
    exact_total = 0
    binary_total = 0
    denom_total = 0
    for aspect in ASPECTS:
        left = frame[f'{left_prefix}_{aspect}'].map(canonical_strength)
        right = frame[f'{right_prefix}_{aspect}'].map(canonical_strength)
        valid = left.isin(STRENGTHS) & right.isin(STRENGTHS)
        n = int(valid.sum())
        exact = int((left[valid] == right[valid]).sum())
        bin_left = left[valid].map(binary_presence)
        bin_right = right[valid].map(binary_presence)
        binary = int((bin_left == bin_right).sum())
        exact_total += exact
        binary_total += binary
        denom_total += n
        by_aspect.append({
            'comparison': comparison,
            'aspect': ASPECT_DISPLAY[aspect],
            'n_cells': n,
            'exact_strength_agreement_pct': round(pct(exact, n), 1),
            'binary_presence_agreement_pct': round(pct(binary, n), 1),
            'exact_strength_agreement_n': exact,
            'binary_presence_agreement_n': binary,
        })
    overall = {
        'comparison': comparison,
        'n_cells': denom_total,
        'exact_strength_agreement_pct': round(pct(exact_total, denom_total), 1),
        'binary_presence_agreement_pct': round(pct(binary_total, denom_total), 1),
        'exact_strength_agreement_n': exact_total,
        'binary_presence_agreement_n': binary_total,
    }
    return overall, pd.DataFrame(by_aspect)

aspect_overall_rows = []
aspect_detail_frames = []
for left, right, comparison in [
    ('a1', 'a2', 'Annotator 1 vs. Annotator 2'),
    ('a1', 'ai', 'Annotator 1 vs. AI aspects'),
    ('a2', 'ai', 'Annotator 2 vs. AI aspects'),
]:
    overall, detail = aspect_pair_agreement(frame, left, right, comparison)
    aspect_overall_rows.append(overall)
    aspect_detail_frames.append(detail)

aspect_overall = pd.DataFrame(aspect_overall_rows)
aspect_by_aspect = pd.concat(aspect_detail_frames, ignore_index=True)
aspect_overall.to_csv(OUT_DIR / 'human_validation_aspect_agreement_overall.csv', index=False)
aspect_by_aspect.to_csv(OUT_DIR / 'human_validation_aspect_agreement_by_aspect.csv', index=False)
aspect_overall


,comparison,n_cells,exact_strength_agreement_pct,binary_presence_agreement_pct,exact_strength_agreement_n,binary_presence_agreement_n
0,Annotator 1 vs. Annotator 2,1800,77.2,81.7,1389,1471
1,Annotator 1 vs. AI aspects,1800,75.2,81.7,1354,1470
2,Annotator 2 vs. AI aspects,1800,72.6,79.8,1306,1437


In [12]:
aspect_by_aspect


,comparison,aspect,n_cells,exact_strength_agreement_pct,binary_presence_agreement_pct,exact_strength_agreement_n,binary_presence_agreement_n
0,Annotator 1 vs. Annotator 2,Depression,300,70.3,75.3,211,226
1,Annotator 1 vs. Annotator 2,Anxiety,300,64.0,72.3,192,217
2,Annotator 1 vs. Annotator 2,Suicidal,300,88.0,90.3,264,271
3,Annotator 1 vs. Annotator 2,Stress,300,60.0,65.7,180,197
4,Annotator 1 vs. Annotator 2,Bipolar,300,93.7,96.3,281,289
5,Annotator 1 vs. Annotator 2,Personality Disorder,300,87.0,90.3,261,271
6,Annotator 1 vs. AI aspects,Depression,300,56.0,66.3,168,199
7,Annotator 1 vs. AI aspects,Anxiety,300,66.3,75.0,199,225
8,Annotator 1 vs. AI aspects,Suicidal,300,86.7,90.7,260,272
9,Annotator 1 vs. AI aspects,Stress,300,58.3,68.3,175,205


In [13]:
# -----------------------------
# Manuscript-style human validation table
# -----------------------------

panel_a = primary_agreement[['comparison', 'exact_agreement', 'agreement_pct', 'cohen_kappa']].copy()
panel_a['exact_agreement_display'] = panel_a['exact_agreement'] + ' (' + panel_a['agreement_pct'].map(lambda x: f'{x:.1f}%') + ')'
panel_a = panel_a[['comparison', 'exact_agreement_display', 'cohen_kappa']]

panel_b = triad.pivot(index='pattern', columns='human_reference', values=['n', 'pct'])
# Stable manuscript order.
pattern_order = ['Human = AI = released', 'Human = released != AI', 'Human = AI != released', 'AI = released != Human', 'All three differ']
panel_b_rows = []
for pat in pattern_order:
    row = {'pattern': pat}
    for ref in ['Annotator 1', 'Annotator 2']:
        n = int(panel_b.loc[pat, ('n', ref)])
        p = float(panel_b.loc[pat, ('pct', ref)])
        row[ref] = f'{n} ({p:.1f}%)'
    panel_b_rows.append(row)
panel_b_display = pd.DataFrame(panel_b_rows)

panel_c = aspect_overall[['comparison', 'exact_strength_agreement_pct', 'binary_presence_agreement_pct']].copy()
panel_c['exact_strength_agreement'] = panel_c['exact_strength_agreement_pct'].map(lambda x: f'{x:.1f}%')
panel_c['binary_presence_agreement'] = panel_c['binary_presence_agreement_pct'].map(lambda x: f'{x:.1f}%')
panel_c = panel_c[['comparison', 'exact_strength_agreement', 'binary_presence_agreement']]

panel_a.to_csv(OUT_DIR / 'manuscript_table_human_validation_panel_a.csv', index=False)
panel_b_display.to_csv(OUT_DIR / 'manuscript_table_human_validation_panel_b.csv', index=False)
panel_c.to_csv(OUT_DIR / 'manuscript_table_human_validation_panel_c.csv', index=False)

print('Panel A: Primary-label agreement')
display(panel_a)
print('Panel B: Three-way primary-label patterns')
display(panel_b_display)
print('Panel C: Aspect-rating agreement')
display(panel_c)


Panel A: Primary-label agreement


,comparison,exact_agreement_display,cohen_kappa
0,Annotator 1 vs. Annotator 2,169/300 (56.3%),0.489
1,Annotator 1 vs. AI label,155/300 (51.7%),0.436
2,Annotator 2 vs. AI label,164/300 (54.7%),0.462
3,Annotator 1 vs. released label,158/300 (52.7%),0.449
4,Annotator 2 vs. released label,145/300 (48.3%),0.401
5,AI label vs. released label,138/300 (46.0%),0.373


Panel B: Three-way primary-label patterns


,pattern,Annotator 1,Annotator 2
0,Human = AI = released,100 (33.3%),97 (32.3%)
1,Human = released != AI,58 (19.3%),48 (16.0%)
2,Human = AI != released,55 (18.3%),67 (22.3%)
3,AI = released != Human,38 (12.7%),41 (13.7%)
4,All three differ,49 (16.3%),47 (15.7%)


Panel C: Aspect-rating agreement


,comparison,exact_strength_agreement,binary_presence_agreement
0,Annotator 1 vs. Annotator 2,77.2%,81.7%
1,Annotator 1 vs. AI aspects,75.2%,81.7%
2,Annotator 2 vs. AI aspects,72.6%,79.8%


## Supplementary Table S2 From Labeling Exploration Confusion Matrix

This section regenerates Supplementary Table S2 from `mental_health_unified_labels_final.csv` using the same original-label vs. `gpt-4o-mini` label orientation as the labeling exploration confusion matrix:

- rows = released `status` label;
- columns = `gpt-4o-mini` label (`u_label`);
- cell values = row-normalized percentages.


In [14]:
# -----------------------------
# Supplementary Table S2 from labeling exploration source
# -----------------------------

def read_labeling_source(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    if 'status' not in df.columns or 'u_label' not in df.columns:
        raise ValueError('Expected columns status and u_label in labeling source')
    df = df.copy()
    df['released_label'] = df['status'].map(canonical_label)
    df['ai_label'] = df['u_label'].map(canonical_label)
    df = df[df['released_label'].isin(LABELS) & df['ai_label'].isin(LABELS)].copy()
    return df


def original_ai_confusion_tables(source: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts = pd.crosstab(source['released_label'], source['ai_label']).reindex(index=LABELS, columns=LABELS, fill_value=0)
    row_pct = counts.div(counts.sum(axis=1), axis=0) * 100.0
    counts.index = [S2_DISPLAY[x] for x in counts.index]
    counts.columns = [S2_DISPLAY[x] for x in counts.columns]
    row_pct.index = [S2_DISPLAY[x] for x in row_pct.index]
    row_pct.columns = [S2_DISPLAY[x] for x in row_pct.columns]
    return counts, row_pct.round(1)

label_source = read_labeling_source(LABEL_SOURCE_PATH)
s2_counts, s2_row_pct = original_ai_confusion_tables(label_source)
s2_counts.to_csv(OUT_DIR / 'table_s2_original_ai_confusion_counts.csv')
s2_row_pct.to_csv(OUT_DIR / 'table_s2_original_ai_confusion_row_percent.csv')

print('Rows:', len(label_source))
print('Supplementary Table S2 row-normalized percentages')
s2_row_pct


Rows: 53043
Supplementary Table S2 row-normalized percentages


,Normal,Depression,Anxiety,Suicidal,Stress,Bipolar,Pers. Dis.
Normal,87.5,4.5,3.1,0.5,4.2,0.0,0.3
Depression,14.7,41.9,12.1,26.6,3.2,0.7,0.8
Anxiety,10.6,1.6,85.0,1.3,1.2,0.1,0.3
Suicidal,10.0,14.3,3.2,70.1,1.5,0.3,0.6
Stress,26.5,11.6,31.3,5.7,22.3,0.2,2.4
Bipolar,25.4,11.6,12.4,8.4,4.1,36.8,1.3
Pers. Dis.,36.1,22.0,28.5,4.9,2.0,0.5,6.1


In [15]:
def s2_latex_rows(row_pct: pd.DataFrame) -> str:
    lines = []
    for idx, row in row_pct.iterrows():
        values = [f'{v:.1f}' for v in row.tolist()]
        diag_pos = list(row_pct.index).index(idx)
        values[diag_pos] = r'\textbf{' + values[diag_pos] + '}'
        lines.append(idx + ' & ' + ' & '.join(values) + r' \\')
    return '\n'.join(lines)

latex_rows = s2_latex_rows(s2_row_pct)
(OUT_DIR / 'table_s2_original_ai_confusion_latex_rows.txt').write_text(latex_rows, encoding='utf-8')
print(latex_rows)


Normal & \textbf{87.5} & 4.5 & 3.1 & 0.5 & 4.2 & 0.0 & 0.3 \\
Depression & 14.7 & \textbf{41.9} & 12.1 & 26.6 & 3.2 & 0.7 & 0.8 \\
Anxiety & 10.6 & 1.6 & \textbf{85.0} & 1.3 & 1.2 & 0.1 & 0.3 \\
Suicidal & 10.0 & 14.3 & 3.2 & \textbf{70.1} & 1.5 & 0.3 & 0.6 \\
Stress & 26.5 & 11.6 & 31.3 & 5.7 & \textbf{22.3} & 0.2 & 2.4 \\
Bipolar & 25.4 & 11.6 & 12.4 & 8.4 & 4.1 & \textbf{36.8} & 1.3 \\
Pers. Dis. & 36.1 & 22.0 & 28.5 & 4.9 & 2.0 & 0.5 & \textbf{6.1} \\


In [16]:
def plot_s2_confusion(row_pct: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(row_pct.values, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks(np.arange(len(row_pct.columns)))
    ax.set_yticks(np.arange(len(row_pct.index)))
    ax.set_xticklabels(row_pct.columns, rotation=45, ha='right')
    ax.set_yticklabels(row_pct.index)
    ax.set_xlabel('gpt-4o-mini label')
    ax.set_ylabel('Released status label')
    ax.set_title('Original vs AI Labels: Row-Normalized Confusion Matrix')
    for i in range(row_pct.shape[0]):
        for j in range(row_pct.shape[1]):
            value = row_pct.iloc[i, j]
            color = 'white' if value >= 50 else 'black'
            ax.text(j, i, f'{value:.1f}', ha='center', va='center', color=color, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Row %')
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.show()

plot_s2_confusion(s2_row_pct, OUT_DIR / 'table_s2_original_ai_confusion_row_percent.png')


In [17]:
# -----------------------------
# Final audit summary
# -----------------------------
summary = {
    'human_validation_rows': int(len(frame)),
    'primary_agreement': primary_agreement.to_dict(orient='records'),
    'triad_counts': triad.to_dict(orient='records'),
    'aspect_agreement_overall': aspect_overall.to_dict(orient='records'),
    'qc_summary': qc_summary.to_dict(orient='records'),
    'qc_issue_count': int(len(qc_issues)),
    'table_s2_rows': int(len(label_source)),
    'outputs_dir': str(OUT_DIR),
}
(OUT_DIR / 'notebook_audit_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


{
  "human_validation_rows": 300,
  "primary_agreement": [
    {
      "comparison": "Annotator 1 vs. Annotator 2",
      "exact_agreement": "169/300",
      "agreement_n": 169,
      "n": 300,
      "agreement_pct": 56.3,
      "cohen_kappa": 0.489
    },
    {
      "comparison": "Annotator 1 vs. AI label",
      "exact_agreement": "155/300",
      "agreement_n": 155,
      "n": 300,
      "agreement_pct": 51.7,
      "cohen_kappa": 0.436
    },
    {
      "comparison": "Annotator 2 vs. AI label",
      "exact_agreement": "164/300",
      "agreement_n": 164,
      "n": 300,
      "agreement_pct": 54.7,
      "cohen_kappa": 0.462
    },
    {
      "comparison": "Annotator 1 vs. released label",
      "exact_agreement": "158/300",
      "agreement_n": 158,
      "n": 300,
      "agreement_pct": 52.7,
      "cohen_kappa": 0.449
    },
    {
      "comparison": "Annotator 2 vs. released label",
      "exact_agreement": "145/300",
      "agreement_n": 145,
      "n": 300,
      "agreeme